In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv")
cols = ["engine_displacement", "horsepower", "vehicle_weight", "model_year", "fuel_efficiency_mpg"]
df = df[cols]
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [2]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [3]:
df.horsepower.median()

np.float64(254.0)

In [4]:
def prepare_split(df, seed):
    n = len(df)
    n_val=int(n * 0.2)
    n_test=int(n * 0.2)
    n_train=n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train+n_val]].reset_index(drop=True)
    df_test = df.iloc[idx[n_train+n_val:]].reset_index(drop=True)

    return df_train, df_val, df_test


def train_lin_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T @ X
    XTX = XTX + r * np.eye(XTX.shape[0])
    w = np.linalg.inv(XTX) @ X.T @ y
    return w[0], w[1:]


def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

base = ["engine_displacement", "horsepower", "vehicle_weight", "model_year"]

In [6]:
df_train, df_val, df_test = prepare_split(df, seed=42)
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

X_train0 = df_train[base].fillna(0).values
X_val0 = df_val[base].fillna(0).values
w0, w0_ = train_lin_reg(X_train0, y_train)
rmse_0 = round(rmse(y_val, w0+ X_val0 @ w0_), 3)

mean_hp = df_train.horsepower.median()
X_train_m = df_train[base].fillna(mean_hp).values
X_val_m = df_val[base].fillna(mean_hp).values
wm, wm_ = train_lin_reg(X_train_m, y_train)
rmse_mean = round(rmse(y_val, wm + X_val_m @ wm_), 3)

print ("RMSE with 0: ", rmse_0)
print ("RMSE with mean: ", rmse_mean)

RMSE with 0:  2.205
RMSE with mean:  2.202


In [7]:
for r in [0.0, 0.01, 0.1, 1.0, 5.0, 10.0, 100.0]:
    w0, w_ = train_lin_reg(X_train0, y_train, r=r)
    score = round(rmse(y_val, w0 + X_val0 @ w_), 4)
    print(r, score)

0.0 2.2053
0.01 2.2058
0.1 2.2241
1.0 2.3492
5.0 2.4094
10.0 2.4195
100.0 2.4292


In [8]:
scores = []
for seed in range(10):
    dtr, dval, dte = prepare_split(df, seed)
    ytr = dtr.fuel_efficiency_mpg.values
    y_val = dval.fuel_efficiency_mpg.values
    Xtr = dtr[base].fillna(0).values
    Xval = dval[base].fillna(0).values
    w0, w_ = train_lin_reg(Xtr, ytr)
    ypred = w0 + Xval @ w_
    scores.append(rmse(y_val, ypred))

round(np.std(scores),3)

np.float64(0.029)

In [9]:
dtr, dval, dte = prepare_split(df, seed=9)
df_full_train = pd.concat([dtr, dval]).reset_index(drop=True)
y_full_train = df_full_train.fuel_efficiency_mpg.values
y_test = dte.fuel_efficiency_mpg.values

X_full_train = df_full_train[base].fillna(0).values
X_test = dte[base].fillna(0).values

w0, w_ = train_lin_reg(X_full_train, y_full_train, r=0.001)
y_pred = w0 + X_test @ w_

round(rmse(y_test, y_pred), 3)

np.float64(2.236)